# 02 — Atribuição direta da diferença de logits

O primeiro notebook mostrou **o que o modelo faz**. Agora perguntamos:

> Quais componentes do residual stream apontam diretamente para o nome correto?

Usaremos **Direct Logit Attribution (DLA)**: projetar a saída de cada componente na direção `correto − incorreto`.

DLA é uma decomposição do resultado, não uma intervenção causal.

In [1]:
import plotly.graph_objects as go
import torch

from ioi_replication import (
    build_ioi_examples,
    load_gpt2_small,
    logits_to_logit_diff,
    select_device,
    tokenize_answers,
)

device = select_device()
model = load_gpt2_small(device)
prompts, answers = build_ioi_examples()
tokens = model.to_tokens(prompts, prepend_bos=True).to(device)
answer_tokens = tokenize_answers(model, answers, device)

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

baseline = logits_to_logit_diff(logits, answer_tokens)
print(f"Dispositivo: {device}")
print(f"Baseline: {baseline.item():+.3f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Dispositivo: cuda
Baseline: +3.552


## 1. Construindo a direção IOI

O unembedding associa cada token a uma direção no residual stream.

Para cada prompt, definimos:

$$d_{IOI}=d_{correto}-d_{incorreto}$$

Se um componente tem produto interno positivo com essa direção, sua contribuição direta favorece o nome correto. Se for negativo, favorece o incorreto.

In [2]:
answer_directions = model.tokens_to_residual_directions(answer_tokens)
correct_directions, incorrect_directions = answer_directions.unbind(dim=1)
logit_diff_directions = correct_directions - incorrect_directions

assert answer_directions.shape == (8, 2, model.cfg.d_model)
assert logit_diff_directions.shape == (8, model.cfg.d_model)
print("answer_directions:", tuple(answer_directions.shape))
print("logit_diff_directions:", tuple(logit_diff_directions.shape))

answer_directions: (8, 2, 768)
logit_diff_directions: (8, 768)


## 2. Reconstruindo a diferença final

Antes do unembedding, o residual final passa pela LayerNorm. Por isso aplicamos ao vetor residual a escala guardada no cache.

Depois disso, o produto interno com $d_{IOI}$ deve reproduzir a diferença de logits observada no primeiro notebook.

In [3]:
final_residual = cache["resid_post", -1][:, -1, :]
scaled_final_residual = cache.apply_ln_to_stack(
    final_residual,
    layer=-1,
    pos_slice=-1,
)
reconstructed = torch.einsum(
    "bd,bd->",
    scaled_final_residual,
    logit_diff_directions,
) / len(prompts)

torch.testing.assert_close(reconstructed, baseline, rtol=1e-4, atol=1e-4)
print(f"Logits:       {baseline.item():+.6f}")
print(f"Reconstrução: {reconstructed.item():+.6f}")

Logits:       +3.551887
Reconstrução: +3.551888


## 3. Decomposição por componentes do residual stream

O residual final é uma soma das embeddings e das saídas de atenção e MLP de cada camada.

Como a projeção final é linear, podemos medir separadamente quanto cada parcela aponta para $d_{IOI}$. A soma das contribuições deve reconstruir o baseline.

In [4]:
residual_stack, residual_labels = cache.decompose_resid(
    layer=-1,
    pos_slice=-1,
    apply_ln=True,
    return_labels=True,
)
residual_contributions = torch.einsum(
    "cbd,bd->cb",
    residual_stack,
    logit_diff_directions,
).mean(dim=1)

torch.testing.assert_close(
    residual_contributions.sum(),
    baseline,
    rtol=1e-4,
    atol=1e-4,
)

figure = go.Figure(
    data=go.Heatmap(
        z=residual_contributions.detach().cpu().numpy()[None, :],
        x=residual_labels,
        y=["contribuição"],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="logit diff",
    )
)
figure.update_layout(
    title="Atribuição direta por componente do residual stream",
    xaxis_title="componente escrito no residual stream",
    yaxis_title="",
    height=320,
)
figure.show()

## 4. Decomposição por head

Agora separamos apenas as 144 heads: `12 camadas × 12 heads`.

No gráfico, cada célula mostra a contribuição direta média de uma head para a diferença `correto − incorreto`.

Uma célula positiva indica alinhamento direto com a resposta correta — não prova que a head seja causalmente necessária.

In [5]:
with torch.no_grad():
    head_stack, head_labels = cache.stack_head_results(
        layer=-1,
        pos_slice=-1,
        apply_ln=True,
        return_labels=True,
    )

head_contributions = torch.einsum(
    "hbd,bd->hb",
    head_stack,
    logit_diff_directions,
).mean(dim=1)
head_contributions = head_contributions.reshape(
    model.cfg.n_layers,
    model.cfg.n_heads,
)

assert head_contributions.shape == (12, 12)

figure = go.Figure(
    data=go.Heatmap(
        z=head_contributions.detach().cpu().numpy(),
        x=[f"H{head}" for head in range(model.cfg.n_heads)],
        y=[f"L{layer}" for layer in range(model.cfg.n_layers)],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="logit diff",
        hovertemplate="camada=%{y}<br>head=%{x}<br>contribuição=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Atribuição direta por attention head",
    xaxis_title="head",
    yaxis_title="camada",
    height=600,
)
figure.show()

## 5. Candidatas a Name Mover Heads

As heads `9.6`, `9.9` e `10.0` têm contribuição direta positiva neste conjunto.

Se estiverem movendo o nome correto para a posição final, esperamos que a atenção emitida de `END` alcance a posição do **IO** — o nome correto que apareceu uma vez no prompt.

In [6]:
candidate_heads = [(9, 6), (9, 9), (10, 0)]
batch_indices = torch.arange(len(prompts), device=device)
io_positions = (tokens == answer_tokens[:, 0, None]).int().argmax(dim=-1)

for layer, head in candidate_heads:
    contribution = head_contributions[layer, head]
    end_attention = cache["pattern", layer][:, head, -1, :]
    io_attention = end_attention[batch_indices, io_positions]
    io_is_top = end_attention.argmax(dim=-1) == io_positions

    assert contribution.item() > 0
    assert io_attention.mean().item() > 0.25
    print(
        f"head {layer}.{head}: "
        f"DLA={contribution.item():+.3f} | "
        f"atenção END→IO={io_attention.mean().item():.3f} | "
        f"IO foi o maior destino em {io_is_top.sum().item()}/8 prompts"
    )

head 9.6: DLA=+1.384 | atenção END→IO=0.694 | IO foi o maior destino em 8/8 prompts
head 9.9: DLA=+2.985 | atenção END→IO=0.817 | IO foi o maior destino em 8/8 prompts
head 10.0: DLA=+0.561 | atenção END→IO=0.363 | IO foi o maior destino em 3/8 prompts


## Conclusão

A atribuição direta encontrou heads que:

1. escrevem em uma direção que favorece o nome correto; e
2. direcionam atenção da posição final para o IO.

Isso sustenta a hipótese de **Name Mover Heads**, especialmente para `9.6` e `9.9` neste conjunto.

Mas ainda é evidência correlacional/decompositiva. Uma head pode ter contribuição direta alta sem ser necessária, ou contribuição direta baixa enquanto transporta informação usada depois. O próximo notebook fará uma intervenção: **activation patching**.